In [1]:
# ============================================================
# IMPORTS
# ============================================================

from ultralytics import YOLO

import cv2
import time
from pathlib import Path

In [2]:
# ============================================================
# PATHS
# ============================================================

PROJECT_ROOT = Path("../")

MODEL_PATH = (PROJECT_ROOT / "models" / "trained" / "ppe_yolo11m_best.pt")

INPUT_VIDEO = (PROJECT_ROOT / "videos" / "input" / "sample_site.mp4")

OUTPUT_VIDEO = (PROJECT_ROOT / "videos" / "output" / "01_detection_demo.mp4")

In [3]:
# ============================================================
# LOAD MODEL
# ============================================================

model = YOLO(str(MODEL_PATH))

print("MODEL LOADED SUCCESSFULLY")

MODEL LOADED SUCCESSFULLY


In [4]:
# ============================================================
# OPEN VIDEO
# ============================================================

cap = cv2.VideoCapture(str(INPUT_VIDEO))

if not cap.isOpened():
    raise ValueError(f"Cannot open video: {INPUT_VIDEO}")

In [5]:
# ============================================================
# VIDEO PROPERTIES
# ============================================================

frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS))

print(f"Resolution : {frame_width} x {frame_height}")
print(f"FPS        : {fps}")

Resolution : 1920 x 1080
FPS        : 30


In [6]:
# ============================================================
# OUTPUT VIDEO WRITER
# ============================================================

OUTPUT_VIDEO.parent.mkdir(
    parents=True,
    exist_ok=True
)

fourcc = cv2.VideoWriter_fourcc(*"mp4v")

out = cv2.VideoWriter(
    str(OUTPUT_VIDEO),
    fourcc,
    fps,
    (frame_width, frame_height)
)

In [7]:
# ============================================================
# PERFORMANCE VARIABLES
# ============================================================

prev_time = 0
frame_count = 0


In [8]:
# ============================================================
# INFERENCE LOOP
# ============================================================

print("\nStarting video inference...\n")

while True:

    success, frame = cap.read()

    if not success:
        print("\nVideo processing completed.")
        break

    # --------------------------------------------------------
    # YOLO INFERENCE
    # --------------------------------------------------------

    results = model.predict(
        source=frame,
        conf=0.50,
        imgsz=960,
        verbose=False,
        device=0
    )

    # --------------------------------------------------------
    # PLOT RESULTS
    # --------------------------------------------------------

    annotated_frame = results[0].plot()

    # --------------------------------------------------------
    # FPS CALCULATION
    # --------------------------------------------------------

    current_time = time.time()

    fps_value = 1 / (current_time - prev_time)

    prev_time = current_time

    # --------------------------------------------------------
    # DISPLAY FPS
    # --------------------------------------------------------

    cv2.putText(
        annotated_frame,
        f"FPS: {fps_value:.2f}",
        (20, 40),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        (0, 255, 0),
        2
    )

    # --------------------------------------------------------
    # DISPLAY FRAME COUNT
    # --------------------------------------------------------

    frame_count += 1

    cv2.putText(
        annotated_frame,
        f"Frame: {frame_count}",
        (20, 80),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        (255, 255, 0),
        2
    )

    # --------------------------------------------------------
    # SHOW VIDEO
    # --------------------------------------------------------

    cv2.imshow(
        "PPE Detection System",
        annotated_frame
    )

    # --------------------------------------------------------
    # SAVE OUTPUT FRAME
    # --------------------------------------------------------

    out.write(annotated_frame)

    # --------------------------------------------------------
    # EXIT KEY
    # --------------------------------------------------------

    key = cv2.waitKey(1)

    if key == ord("q"):
        print("\nInference stopped by user.")
        break


Starting video inference...


Video processing completed.


In [9]:
# ============================================================
# RELEASE RESOURCES
# ============================================================

cap.release()
out.release()
cv2.destroyAllWindows()

In [10]:
# ============================================================
# DONE
# ============================================================

print("=" * 60)
print("VIDEO SAVED SUCCESSFULLY")
print("=" * 60)

print(f"\nOutput Video:\n{OUTPUT_VIDEO}")

VIDEO SAVED SUCCESSFULLY

Output Video:
C:\Users\VANSH\OneDrive - IITRAM\Desktop\Projects\PPE_project\videos\output\ppe_output.mp4
